In [3]:

import os
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# 1. Load the dataset
DATA_FILE = "customer_churn.csv"
RESULTS_FILE = "model_results.csv"

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f"Dataset not found: {DATA_FILE}. "
        "Run this script from your project folder."
    )

df = pd.read_csv(DATA_FILE)

# 2. Validate the target column
target = "churned"

if target not in df.columns:
    raise ValueError(f"Target column '{target}' is missing.")

if df[target].nunique() < 2:
    raise ValueError("The target must contain both churn classes.")

# 3. Separate features and target
# customer_id is an identifier, not a useful model feature.
X = df.drop(columns=[target, "customer_id"], errors="ignore")
y = df[target]

# 4. Split before preprocessing to avoid data leakage
# Stratification may not be reliable with only 12 rows,
# so use it only when each class has enough examples.
class_counts = y.value_counts()
use_stratify = (
    len(class_counts) == 2 and class_counts.min() >= 2
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=42,
    stratify=y if use_stratify else None
)

# 5. Identify numeric and categorical columns
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# 6. Define models
models = {
    "Majority-Class Baseline": DummyClassifier(
        strategy="most_frequent"
    ),
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ])
}

# 7. Train and evaluate on held-out test data
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision_weighted": precision_score(
            y_test, predictions, average="weighted",
            zero_division=0
        ),
        "Recall_weighted": recall_score(
            y_test, predictions, average="weighted",
            zero_division=0
        ),
        "F1_weighted": f1_score(
            y_test, predictions, average="weighted",
            zero_division=0
        )
    })

    print(f"\n{name}")
    print("-" * len(name))
    print(classification_report(
        y_test, predictions, zero_division=0
    ))

# 8. Save results
results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_FILE, index=False)

print("\nEvaluation completed.")
print(results_df.to_string(index=False))
print(f"\nResults saved to {RESULTS_FILE}")
print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

print(
    "\nWarning: This dataset has only 12 rows. "
    "Treat these scores as preliminary, not as proof "
    "that the model will perform well on new customers."
)


Current folder:
c:\Users\Administrator\Desktop\ML-Problem-Framing-Responsible-Data-Card

Dataset loaded successfully!
Dataset shape: (12, 7)

Column names:
['customer_id', 'tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days', 'plan_type', 'churned']

First five rows:


,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0



Missing values in each column:
customer_id          0
tenure_months        0
support_tickets      0
monthly_spend_inr    0
last_login_days      0
plan_type            0
churned              0
dtype: int64

Column data types:
customer_id          object
tenure_months         int64
support_tickets       int64
monthly_spend_inr     int64
last_login_days       int64
plan_type            object
churned               int64
dtype: object

Dataset summary:


,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
count,12,12.000000,12.000000,12.000000,12.000000,12,12.000000
unique,12,NaN,NaN,NaN,NaN,3,NaN
top,C001,NaN,NaN,NaN,NaN,Basic,NaN
freq,1,NaN,NaN,NaN,NaN,5,NaN
mean,NaN,12.166667,2.250000,915.666667,10.000000,NaN,0.416667
std,NaN,9.242524,1.602555,332.574895,9.095453,NaN,0.514929
min,NaN,1.000000,0.000000,499.000000,1.000000,NaN,0.000000
25%,NaN,5.500000,1.000000,724.000000,2.750000,NaN,0.000000
50%,NaN,9.500000,2.000000,899.000000,7.000000,NaN,0.000000
75%,NaN,18.500000,3.250000,1074.000000,14.750000,NaN,1.000000
